# Análisis de calidad de los datos oficiales

## tl;dr

Las fuentes contienen volumen suficiente para una demostración completa y
errores intencionales controlados. No reducimos Bronze: Silver explica cada
fila y Gold conserva los conteos y métricas de negocio de referencia.

In [ ]:
from pathlib import Path
import json
import re
import pandas as pd
from IPython.display import display

RAIZ = Path.cwd().resolve()
if not (RAIZ / "datos" / "entrada").exists():
    RAIZ = RAIZ.parent
ENTRADA = RAIZ / "datos" / "entrada"
nombres = ["sucursales", "productos", "clientes", "cambios_cliente",
           "cambios_producto", "ventas", "devoluciones", "stock_diario"]
tablas = {
    nombre: pd.read_csv(ENTRADA / f"{nombre}.csv", dtype=str, keep_default_na=False)
    for nombre in nombres
}

## Perfil y grano

In [ ]:
perfil = pd.DataFrame([
    {
        "fuente": nombre,
        "filas": len(datos),
        "columnas": len(datos.columns),
        "duplicados_exactos": int(datos.duplicated().sum()),
    }
    for nombre, datos in tablas.items()
])
display(perfil)

## Identidad, referencias y dominios

In [ ]:
clientes = tablas["clientes"].assign(
    id_norm=lambda x: x.id_cliente.str.strip().str.upper(),
    correo_norm=lambda x: x.correo.str.strip().str.lower(),
)
from src.comun.correos import correo_valido
ventas = tablas["ventas"]
hallazgos = pd.DataFrame([
    ("clientes duplicados normalizados", int(clientes.duplicated("id_norm", keep=False).sum())),
    ("correos duplicados normalizados", int(clientes.duplicated("correo_norm", keep=False).sum())),
    ("correos con formato inválido", int((~clientes.correo_norm.map(correo_valido)).sum())),
    ("ventas con cliente desconocido", int((~ventas.id_cliente.str.strip().str.upper().isin(set(clientes.id_norm))).sum())),
    ("ventas con producto desconocido", int((~ventas.codigo_producto.str.strip().str.upper().isin(set(tablas["productos"].codigo_producto.str.strip().str.upper()))).sum())),
    ("ventas con sucursal desconocida", int((~ventas.codigo_sucursal.str.strip().str.upper().isin(set(tablas["sucursales"].codigo_sucursal.str.strip().str.upper()))).sum())),
], columns=["control", "filas"])
display(hallazgos)

## Fechas y medidas de ventas

In [ ]:
fecha = pd.to_datetime(ventas.fecha_venta, errors="coerce")
medidas = {
    "fecha_venta_invalida": int(fecha.isna().sum()),
    "cantidad_no_positiva": int((pd.to_numeric(ventas.cantidad, errors="coerce") <= 0).sum()),
    "precio_no_numerico": int(pd.to_numeric(ventas.precio_unitario, errors="coerce").isna().sum()),
    "descuento_fuera_rango": int((~pd.to_numeric(ventas.porcentaje_descuento, errors="coerce").between(0, 100)).sum()),
}
display(pd.Series(medidas, name="filas").to_frame())
print("Período válido:", fecha.min().date(), "a", fecha.max().date())

## Auditoría Silver ejecutada

In [ ]:
ruta_auditoria = RAIZ / "resultados" / "silver" / "auditoria_silver.json"
auditoria = json.loads(ruta_auditoria.read_text(encoding="utf-8"))
display(pd.DataFrame(auditoria["conciliacion"]))
rechazos = pd.DataFrame(auditoria["rechazos"])
display(rechazos[["fuente", "motivo"]].value_counts().reset_index(name="filas"))
assert all(fila["diferencia"] == 0 for fila in auditoria["conciliacion"])

## Muestra legible de correos

Esta muestra separada usa nombres ficticios plausibles. Sirve para enseñar
formato, vacío y duplicidad sin modificar los CSV oficiales ni sus métricas.

In [ ]:
muestra = pd.read_csv(
    RAIZ / "datos" / "muestra_calidad" / "clientes_muestra.csv",
    dtype=str,
    keep_default_na=False,
)
muestra["correo_normalizado"] = muestra.correo.str.strip().str.lower()
muestra["formato_valido"] = muestra.correo_normalizado.map(
    correo_valido
)
muestra["correo_duplicado"] = muestra.correo_normalizado.ne("") & muestra.correo_normalizado.duplicated(keep=False)
display(muestra[["id_cliente", "nombre_cliente", "correo", "formato_valido", "correo_duplicado"]])
display(pd.Series({
    "formato_valido": int(muestra.formato_valido.sum()),
    "formato_invalido_o_vacio": int((~muestra.formato_valido).sum()),
    "filas_con_correo_duplicado": int(muestra.correo_duplicado.sum()),
}, name="filas").to_frame())

## Takeaways

- El volumen es manejable y cubre suficientes casos de calidad; no se reduce.
- Los correos son sintéticos, con dominios variados y ocho casos inválidos deliberados; Silver valida su formato sin verificar buzones.
- Los errores intencionales permanecen en Bronze y quedan explicados por Silver.
- La separación de capas permite enseñar limpieza antes de construir dimensiones.